# PS S6E6 - 018 shared003 XGBoost as-is

Experiment: `exp_20260606_018_shared003_xgb_as_is`

Purpose:
- Extract only the XGBoost part from `shared003.ipynb`.
- Omit the other-submission / external blend part.
- Reproduce the shared003 XGB branch as-is as much as practical.
- Save OOF/test probability `.npy` files with the full `exp_*` id in filenames.
- Save probabilities in the project-standard class order: `GALAXY, QSO, STAR`.

Important:
- shared003 internally maps classes as `GALAXY=0, STAR=1, QSO=2`.
- Existing S6E6 assets use `GALAXY, QSO, STAR`.
- Therefore this notebook trains/evaluates in shared003 internal order, then reorders OOF/pred probabilities before saving `.npy`.
- Do not skip the reorder step, or future blends will silently mix QSO and STAR columns.

In [1]:
# ============================================================
# 0. Optional installs
# ============================================================

# Uncomment if Kaggle environment does not have the required versions.
# !pip install -q --upgrade scikit-learn==1.7.2 xgboost==3.1.3 category_encoders

In [2]:
# ============================================================
# 1. Imports / Config
# ============================================================

import os
import gc
import json
import math
import random
import warnings
from pathlib import Path
from itertools import combinations
from copy import deepcopy

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

try:
    import torch
except Exception:
    torch = None

try:
    import yaml
except Exception:
    yaml = None


COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260606_018_shared003_xgb_as_is"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
ORIGINAL_PATH = Path("/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET = "class"
ID_COL = "id"
N_SPLITS = 5
EARLY_STOP = 100

# shared003 internal order
SHARED_CLASS_ORDER = ["GALAXY", "STAR", "QSO"]
SHARED_MAPPING = {"GALAXY": 0, "STAR": 1, "QSO": 2}
SHARED_INV_MAPPING = {v: k for k, v in SHARED_MAPPING.items()}

# project-standard order used by previous S6E6 npy assets
STANDARD_CLASS_ORDER = ["GALAXY", "QSO", "STAR"]
STANDARD_MAPPING = {"GALAXY": 0, "QSO": 1, "STAR": 2}
STANDARD_REORDER_IDX = [SHARED_MAPPING[c] for c in STANDARD_CLASS_ORDER]  # shared -> standard

XGB_DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

OOF_NPY = OUTDIR / "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy"
PRED_NPY = OUTDIR / "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy"

def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("XGB_DEVICE:", XGB_DEVICE)
print("STANDARD_REORDER_IDX:", STANDARD_REORDER_IDX)

EXP_ID: exp_20260606_018_shared003_xgb_as_is
OUTDIR: /kaggle/working/exp_20260606_018_shared003_xgb_as_is
XGB_DEVICE: cuda
STANDARD_REORDER_IDX: [0, 2, 1]


In [3]:
# ============================================================
# 2. Load data
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH, ORIGINAL_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train_raw = pd.read_csv(TRAIN_PATH, index_col=ID_COL)
test_raw = pd.read_csv(TEST_PATH, index_col=ID_COL)
sample = pd.read_csv(SAMPLE_SUB_PATH)
orig_raw = pd.read_csv(ORIGINAL_PATH)

train_id = train_raw.index.to_numpy()
test_id = test_raw.index.to_numpy()

train = train_raw.copy()
test = test_raw.copy()
orig = orig_raw.copy()

train[TARGET] = train[TARGET].map(SHARED_MAPPING).astype(int)
orig[TARGET] = orig[TARGET].map(SHARED_MAPPING).astype(int)

y = train[TARGET]
X_base = train.drop(TARGET, axis=1)

print("train:", train.shape)
print("test:", test.shape)
print("orig:", orig.shape)
print("target counts:")
display(y.map(SHARED_INV_MAPPING).value_counts())

train: (577347, 11)
test: (247435, 10)
orig: (100000, 18)
target counts:


class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64

In [4]:
# ============================================================
# 3. shared003 feature helper functions
# ============================================================

def spectral_type(g, r):
    return pd.cut(
        r - g,
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=["M", "G/K", "A/F", "O/B"]
    ).astype(str)


def galaxy_population(u, r):
    return pd.cut(
        u - r,
        [-np.inf, 2.2, np.inf],
        labels=["Blue_Cloud", "Red_Sequence"]
    ).astype(str)


class Preprocessing:
    def __init__(self, target):
        self.target = target
        self._fitted = False

    def fit_transform(self, X_train: pd.DataFrame, y_train: pd.Series, orig: pd.DataFrame = None):
        self.fit(X_train, y_train, orig=orig)
        return self.transform(X_train)

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, orig: pd.DataFrame = None):
        self.X_train_ = X_train.copy()
        self.y_train_ = y_train.copy()

        self.num_features_ = self.X_train_.select_dtypes(exclude=["object", "bool", "category"]).columns.tolist()
        self.cat_features_ = self.X_train_.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

        self._orig_stats = {}

        if orig is not None:
            self.orig_ = orig.copy()
            self._fit_orig_target_stats(self.cat_features_ + self.num_features_)

        self._fit_feature_engineering(self.X_train_)
        X_fe_train = self._apply_feature_engineering(self.X_train_)

        self._fit_frequency_encoding(X_fe_train)
        X_fe_train = self._apply_frequency_encoding(X_fe_train)

        X_fe_train = self._finalize_types(X_fe_train, fit=True)

        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame):
        if not self._fitted:
            raise RuntimeError("Preprocessing is not fitted. Call fit() first.")

        X = X.copy()
        X = self._apply_orig_target_stats(X)
        X = self._apply_feature_engineering(X)
        X = self._apply_frequency_encoding(X)
        X = self._finalize_types(X, fit=False)

        for c in self.cat_features:
            X[c] = X[c].astype(object).fillna("NaN").astype("category")

        return X, self.cat_features, self.num_features

    def _fit_orig_target_stats(self, cols):
        self._orig_global_mean = float(self.orig_[self.target].mean())
        for c in cols:
            if c in self.orig_.columns:
                self._orig_stats[c] = self.orig_.groupby(c, observed=False)[self.target].mean()
            else:
                self._orig_stats[c] = pd.Series(dtype=float)

    def _apply_orig_target_stats(self, df):
        df = df.copy()
        for c, ser in self._orig_stats.items():
            col = f"{c}_org_mean"
            if c in df.columns and not ser.empty:
                df[col] = df[c].map(ser).astype(float).fillna(self._orig_global_mean)
            else:
                df[col] = self._orig_global_mean
        return df

    def _fit_feature_engineering(self, X_train):
        X_train = X_train.copy()

        self._highcard_num = [c for c in self.num_features_ if X_train[c].nunique(dropna=False) > 20]
        self.category_map = {}

        self._bin_edges = {}
        self.bin_features = []
        for c in self._highcard_num:
            _, bins = pd.qcut(X_train[c], q=10, retbins=True, duplicates="drop")
            bins = np.unique(bins)
            if len(bins) >= 3:
                self._bin_edges[c] = bins
                self.bin_features.append(f"{c}_bin")

        self.num_to_cat = []
        for col in self.num_features_:
            cat_name = f"{col}_cat"
            codes, uniques = np.floor(X_train[col]).factorize()
            self.category_map[col] = uniques
            X_train[cat_name] = codes.astype(str)
            self.num_to_cat.append(cat_name)

    def _apply_feature_engineering(self, df):
        df = df.copy()

        for c, bins in self._bin_edges.items():
            out = pd.cut(df[c], bins=bins, labels=False, include_lowest=True, duplicates="drop")
            df[f"{c}_bin"] = out.astype("category")

        for col in self.num_features_:
            cat_name = f"{col}_cat"
            uniques = self.category_map.get(col, [])
            code_map = {cat: i for i, cat in enumerate(uniques)}
            mapped = np.floor(df[col]).map(code_map).fillna(-1).astype("int32")
            df[cat_name] = mapped.astype(str)

        for c in self.num_features_:
            df[f"Log_{c}"] = np.log1p(df[c])
            df[f"{c}_sq"] = df[c] ** 2
            df[f"{c}_sqrt"] = df[c] ** 0.5

        for c1, c2 in combinations(self.num_features_, 2):
            df[f"{c1}_{c2}_ratio"] = df[c1] / (df[c2] + 1e-6)
            df[f"{c1}_-_{c2}"] = df[c1] - df[c2]

        drop = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
        if drop:
            df.drop(drop, axis=1, inplace=True)

        return df

    def _fit_frequency_encoding(self, X):
        all_cats = list(dict.fromkeys(self.cat_features_ + self.num_to_cat))
        self._fe_cols = [c for c in all_cats if c in X.columns]

        self._freq_encodings = {}
        for c in self._fe_cols:
            self._freq_encodings[c] = X[c].value_counts(normalize=True).to_dict()

    def _apply_frequency_encoding(self, X):
        X = X.copy()
        for c in getattr(self, "_fe_cols", []):
            mapping = self._freq_encodings.get(c, {})
            if c in X.columns:
                X[f"{c}_fe"] = X[c].map(mapping).astype(float).fillna(0.0)
            else:
                X[f"{c}_fe"] = 0.0
        return X

    def _finalize_types(self, X, fit=False):
        self.num_features = X.select_dtypes(exclude=["object", "bool", "category"]).columns.tolist()
        self.cat_features = X.select_dtypes(include=["object", "bool", "category"]).columns.tolist()
        return X

In [5]:
# ============================================================
# 4. Preprocess features
# ============================================================

# shared003 adds these two derived categorical features to original first,
# then uses original target means for columns matching competition feature columns.
orig["spectral_type"] = spectral_type(orig["g"], orig["r"])
orig["galaxy_population"] = galaxy_population(orig["u"], orig["r"])

# Match shared003 behavior: use original rows restricted to X_base columns + target.
orig_for_stats = orig[X_base.columns.to_list() + [TARGET]]

pre = Preprocessing(TARGET)
X, cat_features, num_features = pre.fit_transform(X_base, y, orig_for_stats)
X_test, _, _ = pre.transform(test)

feature_info = {
    "n_features": int(X.shape[1]),
    "n_num_features": int(len(num_features)),
    "n_cat_features": int(len(cat_features)),
    "num_features": num_features,
    "cat_features": cat_features,
    "standard_class_order": STANDARD_CLASS_ORDER,
    "shared_internal_class_order": SHARED_CLASS_ORDER,
    "standard_reorder_idx": STANDARD_REORDER_IDX,
}

print("X:", X.shape)
print("X_test:", X_test.shape)
print("num_features:", len(num_features))
print("cat_features:", len(cat_features))
display(X.head())

X: (577347, 120)
X_test: (247435, 120)
num_features: 102
cat_features: 18


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,spectral_type_fe,galaxy_population_fe,alpha_cat_fe,delta_cat_fe,u_cat_fe,g_cat_fe,r_cat_fe,i_cat_fe,z_cat_fe,redshift_cat_fe
id,,,,,,,,,,,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,...,0.525374,0.553506,0.006088,0.011924,0.077371,0.254620,0.302827,0.283462,0.181920,0.778371
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,...,0.525374,0.553506,0.006015,0.017710,0.118963,0.073975,0.132494,0.121511,0.115215,0.778371
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,...,0.075095,0.446494,0.004621,0.022103,0.146977,0.254620,0.237855,0.214194,0.166661,0.058902
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,...,0.525374,0.553506,0.007342,0.007146,0.171673,0.254620,0.124831,0.124227,0.109780,0.778371
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,...,0.525374,0.553506,0.007883,0.016919,0.146977,0.073975,0.087040,0.121511,0.109780,0.778371


In [6]:
# ============================================================
# 5. XGBoost model config
# ============================================================

xgb_params = {
    "n_estimators": 50000,
    "learning_rate": 0.01,
    "max_depth": 6,
    "random_state": SEED,
    "objective": "multi:softprob",
    "num_class": len(SHARED_CLASS_ORDER),
    "eval_metric": "mlogloss",
    "enable_categorical": True,
    "device": XGB_DEVICE,
    "early_stopping_rounds": EARLY_STOP,
}

print(xgb_params)

{'n_estimators': 50000, 'learning_rate': 0.01, 'max_depth': 6, 'random_state': 42, 'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss', 'enable_categorical': True, 'device': 'cuda', 'early_stopping_rounds': 100}


In [7]:
# ============================================================
# 6. Train XGBoost only
# ============================================================

def score_metric(y_true, proba):
    pred = np.argmax(proba, axis=1)
    return balanced_accuracy_score(y_true, pred)


def class_recalls(y_true, pred, labels):
    out = {}
    for i, label in enumerate(labels):
        mask = y_true == i
        out[f"recall_{label}"] = float((pred[mask] == i).mean()) if mask.any() else float("nan")
    return out


folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_shared = np.zeros((len(X), len(SHARED_CLASS_ORDER)), dtype=np.float32)
pred_shared = np.zeros((len(X_test), len(SHARED_CLASS_ORDER)), dtype=np.float32)

sample_weights = compute_sample_weight(class_weight="balanced", y=y)
fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(folds.split(X, y), 1):
    print("=" * 40)
    print(f"Fold {fold}/{N_SPLITS}")

    X_train = X.iloc[tr_idx].copy()
    y_train = y.iloc[tr_idx].copy()
    X_val = X.iloc[va_idx].copy()
    y_val = y.iloc[va_idx].copy()
    X_tst = X_test.copy()
    sw = sample_weights[tr_idx]

    def _rename(arr):
        return pd.DataFrame(arr, columns=[f"te_{i}" for i in range(arr.shape[1])])

    # Fold-safe multiclass TargetEncoder on categorical features.
    te = TargetEncoder(random_state=SEED, shuffle=True, cv=N_SPLITS, smooth="auto", target_type="multiclass")
    X_train_enc = _rename(te.fit_transform(X_train[cat_features].reset_index(drop=True), y_train).astype("float32"))
    X_val_enc = _rename(te.transform(X_val[cat_features].reset_index(drop=True)).astype("float32"))
    X_test_enc = _rename(te.transform(X_tst[cat_features].reset_index(drop=True)).astype("float32"))

    X_train_model = pd.concat([X_train.reset_index(drop=True), X_train_enc], axis=1).drop(cat_features, axis=1)
    X_val_model = pd.concat([X_val.reset_index(drop=True), X_val_enc], axis=1).drop(cat_features, axis=1)
    X_test_model = pd.concat([X_tst.reset_index(drop=True), X_test_enc], axis=1).drop(cat_features, axis=1)

    model = XGBClassifier(**xgb_params)
    model.fit(
        X_train_model,
        y_train,
        sample_weight=sw,
        eval_set=[(X_val_model, y_val)],
        verbose=False,
    )

    val_proba = model.predict_proba(X_val_model).astype(np.float32)
    test_proba = model.predict_proba(X_test_model).astype(np.float32)

    oof_shared[va_idx] = val_proba
    pred_shared += test_proba / N_SPLITS

    fold_score = score_metric(y_val, val_proba)
    val_pred = val_proba.argmax(axis=1)
    row = {
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        "best_iteration": int(getattr(model, "best_iteration", -1) if getattr(model, "best_iteration", None) is not None else -1),
        "n_train": int(len(tr_idx)),
        "n_valid": int(len(va_idx)),
    }
    row.update(class_recalls(y_val.to_numpy(), val_pred, SHARED_CLASS_ORDER))
    fold_rows.append(row)

    print(f"Fold {fold} BA: {fold_score:.8f}")
    print("best_iteration:", row["best_iteration"])

    del model, X_train_model, X_val_model, X_test_model, X_train_enc, X_val_enc, X_test_enc
    gc.collect()

fold_scores = pd.DataFrame(fold_rows)
raw_cv_shared_order = score_metric(y, oof_shared)

print("=" * 40)
print("OOF CV shared order:", raw_cv_shared_order)
display(fold_scores)
print(classification_report(y, oof_shared.argmax(axis=1), target_names=SHARED_CLASS_ORDER))

Fold 1/5
Fold 1 BA: 0.96517898
best_iteration: 13962
Fold 2/5
Fold 2 BA: 0.96521975
best_iteration: 12457
Fold 3/5
Fold 3 BA: 0.96552410
best_iteration: 11633
Fold 4/5
Fold 4 BA: 0.96505063
best_iteration: 11684
Fold 5/5
Fold 5 BA: 0.96506650
best_iteration: 13686
OOF CV shared order: 0.965207986418628


,fold,balanced_accuracy,best_iteration,n_train,n_valid,recall_GALAXY,recall_STAR,recall_QSO
0,1,0.965179,13962,461877,115470,0.968753,0.954186,0.972598
1,2,0.965220,12457,461877,115470,0.965839,0.956966,0.972854
2,3,0.965524,11633,461878,115469,0.965905,0.958837,0.971830
3,4,0.965051,11684,461878,115469,0.967442,0.956180,0.971530
4,5,0.965066,13686,461878,115469,0.966409,0.957389,0.971402


              precision    recall  f1-score   support

      GALAXY       0.99      0.97      0.98    377480
        STAR       0.90      0.96      0.93     82724
         QSO       0.96      0.97      0.97    117143

    accuracy                           0.97    577347
   macro avg       0.95      0.97      0.96    577347
weighted avg       0.97      0.97      0.97    577347



In [8]:
# ============================================================
# 7. Reorder probability columns to project-standard order
# ============================================================

# shared internal order: GALAXY, STAR, QSO
# project-standard order: GALAXY, QSO, STAR
oof_standard = oof_shared[:, STANDARD_REORDER_IDX].astype(np.float32)
pred_standard = pred_shared[:, STANDARD_REORDER_IDX].astype(np.float32)

# For standard-order evaluation, remap y accordingly.
y_label = pd.Series(y).map(SHARED_INV_MAPPING).values
y_standard = pd.Series(y_label).map(STANDARD_MAPPING).astype(int).values

raw_cv_standard_order = balanced_accuracy_score(y_standard, oof_standard.argmax(axis=1))

print("shared order CV :", raw_cv_shared_order)
print("standard order CV:", raw_cv_standard_order)
assert abs(raw_cv_shared_order - raw_cv_standard_order) < 1e-12

print("standard order classification report")
print(classification_report(y_standard, oof_standard.argmax(axis=1), target_names=STANDARD_CLASS_ORDER))

shared order CV : 0.965207986418628
standard order CV: 0.965207986418628
standard order classification report
              precision    recall  f1-score   support

      GALAXY       0.99      0.97      0.98    377480
         QSO       0.96      0.97      0.97    117143
        STAR       0.90      0.96      0.93     82724

    accuracy                           0.97    577347
   macro avg       0.95      0.97      0.96    577347
weighted avg       0.97      0.97      0.97    577347



In [9]:
# ============================================================
# 8. Submission
# ============================================================

test_pred_standard = pred_standard.argmax(axis=1)
test_labels = pd.Series(test_pred_standard).map({v: k for k, v in STANDARD_MAPPING.items()}).values

submission = pd.DataFrame({
    ID_COL: sample[ID_COL].values,
    TARGET: test_labels,
})

assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

submission_path = OUTDIR / "submission_shared003_xgb_as_is.csv"
submission.to_csv(submission_path, index=False)

print(submission[TARGET].value_counts(normalize=True))
display(submission.head())
print("saved:", submission_path)

class
GALAXY    0.642314
QSO       0.205581
STAR      0.152105
Name: proportion, dtype: float64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


saved: /kaggle/working/exp_20260606_018_shared003_xgb_as_is/submission_shared003_xgb_as_is.csv


In [10]:
# ============================================================
# 9. Save artifacts
# ============================================================

np.save(OOF_NPY, oof_standard)
np.save(PRED_NPY, pred_standard)

# Optional audit copies in shared internal order.
np.save(OUTDIR / "oof_exp_20260606_018_shared003_xgb_as_is_shared_order_proba.npy", oof_shared.astype(np.float32))
np.save(OUTDIR / "pred_exp_20260606_018_shared003_xgb_as_is_shared_order_proba.npy", pred_shared.astype(np.float32))

oof_df = pd.DataFrame({ID_COL: train_id})
pred_df = pd.DataFrame({ID_COL: test_id})

for i, cls in enumerate(STANDARD_CLASS_ORDER):
    oof_df[f"proba_{cls}"] = oof_standard[:, i]
    pred_df[f"proba_{cls}"] = pred_standard[:, i]

oof_df["y_true"] = y_label
oof_df["y_true_standard_id"] = y_standard
oof_df["y_pred_standard_id"] = oof_standard.argmax(axis=1)
oof_df["y_pred"] = pd.Series(oof_standard.argmax(axis=1)).map({v: k for k, v in STANDARD_MAPPING.items()}).values

oof_df.to_csv(OUTDIR / "oof_shared003_xgb_as_is.csv", index=False)
pred_df.to_csv(OUTDIR / "pred_shared003_xgb_as_is.csv", index=False)

fold_scores.to_csv(OUTDIR / "fold_scores.csv", index=False)
save_json(feature_info, OUTDIR / "feature_info.json")

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "model": "XGBClassifier",
    "source": "shared003_xgb_only_as_is",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "cv_score": float(raw_cv_standard_order),
    "cv_score_shared_order": float(raw_cv_shared_order),
    "fold_scores": fold_scores.to_dict(orient="records"),
    "class_order_saved_npy": STANDARD_CLASS_ORDER,
    "class_order_shared_internal": SHARED_CLASS_ORDER,
    "standard_reorder_idx": STANDARD_REORDER_IDX,
    "label_mapping_saved_npy": STANDARD_MAPPING,
    "label_mapping_shared_internal": SHARED_MAPPING,
    "params": xgb_params,
    "features": feature_info,
    "use_original": True,
    "original_usage": "original target mean features only, no original row concat",
    "use_fe": True,
    "fe_family": "shared003_xgb_fe_as_is",
    "use_bias_search": False,
    "external_submission_blend": False,
    "submission_path": str(submission_path),
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
}
save_json(cv_result, OUTDIR / "cv_result.json")

print("Saved artifacts:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved artifacts:
 - cv_result.json
 - feature_info.json
 - fold_scores.csv
 - oof_exp_20260606_018_shared003_xgb_as_is_proba.npy
 - oof_exp_20260606_018_shared003_xgb_as_is_shared_order_proba.npy
 - oof_shared003_xgb_as_is.csv
 - pred_exp_20260606_018_shared003_xgb_as_is_proba.npy
 - pred_exp_20260606_018_shared003_xgb_as_is_shared_order_proba.npy
 - pred_shared003_xgb_as_is.csv
 - submission_shared003_xgb_as_is.csv


In [11]:
# ============================================================
# 10. Update registry / memo
# ============================================================

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "XGBoost",
    "feature_family": "shared003_xgb_fe_as_is",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(raw_cv_standard_order),
    "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    "public_lb": np.nan,
    "use_original": True,
    "use_fe": True,
    "use_bias_search": False,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(submission_path),
    "role": "shared_code_xgb_material_candidate",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_corr",
    "notes": (
        "shared003 XGBoost-only as-is. External submission/blend part omitted. "
        "OOF/pred npy saved in standard class order GALAXY,QSO,STAR."
    ),
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "shared003 XGBoost as-is",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "XGBClassifier",
        "created_at": "2026-06-06",
    },
    "objective": {
        "summary": (
            "Extract and run only the XGBoost part from shared003. "
            "Omit external submission/blending. Save OOF/pred probabilities with exp_id filenames "
            "for later correlation and blend checks against 017 current best."
        ),
        "success_criteria": [
            "run shared003 XGB-only",
            "omit external submission blend",
            "save OOF proba npy with exp_id in filename",
            "save test pred proba npy with exp_id in filename",
            "save probabilities in standard class order GALAXY,QSO,STAR",
            "save submission",
            "save cv_result",
            "save fold_scores",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "original_path": str(ORIGINAL_PATH),
        "target_col": TARGET,
        "id_col": ID_COL,
        "use_original": True,
        "original_usage": "original target mean features only, no original row concat",
    },
    "features": {
        "feature_family": "shared003_xgb_fe_as_is",
        "feature_info": feature_info,
        "target_encoder_safety": (
            "TargetEncoder is fit inside each outer fold on train fold categorical features and labels, "
            "then transforms validation and test."
        ),
        "class_order_note": (
            "shared003 internal order is GALAXY,STAR,QSO. "
            "Saved npy order is converted to GALAXY,QSO,STAR for compatibility with previous assets."
        ),
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "score": float(raw_cv_standard_order),
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": "oof_shared003_xgb_as_is.csv",
        "pred_csv": "pred_shared003_xgb_as_is.csv",
        "submission": "submission_shared003_xgb_as_is.csv",
        "cv_result": "cv_result.json",
        "fold_scores": "fold_scores.csv",
        "feature_info": "feature_info.json",
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb_and_corr",
        "reason": (
            "shared003 XGB is a different family from 017 RealMLP. "
            "Adoption depends on reproduced CV, Public LB, and OOF correlation/blend behavior against 017."
        ),
        "next_action": [
            "Submit submission_shared003_xgb_as_is.csv",
            "Record Public LB",
            "Add OOF/pred npy to npy-files dataset",
            "Compare OOF correlation against 017 current best",
            "Blend with 017 if correlation and Public LB are favorable",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())

registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/exp_20260606_018_shared003_xgb_as_is/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260606_018_shared003_xgb_as_is,XGBoost,shared003_xgb_fe_as_is,balanced_accuracy_score,0.965208,0.000171,NaN,True,True,False,/kaggle/working/exp_20260606_018_shared003_xgb...,/kaggle/working/exp_20260606_018_shared003_xgb...,/kaggle/working/exp_20260606_018_shared003_xgb...,shared_code_xgb_material_candidate,completed,pending_public_lb_and_corr,shared003 XGBoost-only as-is. External submiss...
